In [1]:
from __future__ import print_function
import psutil
print(psutil.cpu_percent())
print(psutil.virtual_memory())  # physical memory usage
print('memory % used:', psutil.virtual_memory()[2])

0.9
svmem(total=2164166295552, available=2138722746368, percent=1.2, used=15355158528, free=2106691518464, active=4422381568, inactive=43433426944, buffers=1436643328, cached=40682975232, shared=13864960, slab=5139468288)
memory % used: 1.2


In [2]:
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Activation, Dense, Dropout, Flatten, Input, Embedding, Conv1D
from tensorflow.keras import Sequential
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from tensorflow.keras.utils import plot_model
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras import optimizers
from sklearn.metrics import classification_report
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

2025-11-16 03:53:11.979767: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-16 03:53:12.068080: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-16 03:53:14.083823: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [4]:
# Import all libraries
import numpy as np
import pandas as pd
import pickle
import sys
# import cx_Oracle
import h5py
import timeit
from sklearn import preprocessing
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
# import xgboost as xgb
import pickle
from joblib import dump, load
from sklearn.utils import class_weight
import oracledb
import gzip
import pickle
# import matplotlib.pyplot as plt

# Ensure CPU Usage
# tf.debugging.set_log_device_placement(True)



# Set Display and Printing options
pd.options.display.max_columns=None
pd.set_option('display.max_rows', 500000)
pd.set_option('display.width', 80)
np.set_printoptions(threshold=sys.maxsize)
np.set_printoptions(suppress=True)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [5]:
def read_oracle_data(query, user, password, dsn=None, host=None, port=None, service_name=None, encoding="UTF-8"):
    """
    Read data from Oracle database using oracledb
    
    Args:
        query (str): SQL query to execute
        user (str): Database username
        password (str): Database password
        dsn (str): Data Source Name (optional if host, port, service_name provided)
        host (str): Database host (optional if dsn provided)
        port (int): Database port (optional if dsn provided)
        service_name (str): Database service name (optional if dsn provided)
        encoding (str): Character encoding (default: UTF-8)
    
    Returns:
        pd.DataFrame: Query results as DataFrame
    """
    try:
        # Create connection
        if dsn:
            # Use DSN if provided
            conn = oracledb.connect(user=user, password=password, dsn=dsn)
        else:
            # Build DSN from host, port, service_name
            dsn_string = f"(DESCRIPTION=(ADDRESS=(PROTOCOL=TCP)(HOST={host})(PORT={port}))(CONNECT_DATA=(SERVER=dedicated)(SERVICE_NAME={service_name})))"
            conn = oracledb.connect(user=user, password=password, dsn=dsn_string)
        
        # Execute query and return DataFrame
        df = pd.read_sql(query, con=conn)
        
        # Close connection
        conn.close()
        
        return df
        
    except Exception as e:
        print(f"Error connecting to Oracle database: {e}")
        return None


# Example usage with your config:
config = {
    "user": "GPBI_MLOPS",
    "password": "Manag3r$Mlops", 
    "dsn": "(DESCRIPTION=(ADDRESS=(PROTOCOL=TCP)(HOST=10.15.75.12)(PORT=1252))(CONNECT_DATA=(SERVER=dedicated)(SERVICE_NAME=EDWPROD)))",
    "host": "10.15.75.12",
    "port": 1252,
    "service_name": "edw-100g-scan",
    "encoding": "UTF-8"
}




In [12]:
# Step 2: Execute the SQL queries to retrieve data
query1 = "SELECT * FROM TBL_PL_BASE_01"
query2 = "SELECT * FROM TBL_PL_BASE_02"
query3 = "SELECT * FROM TBL_PL_BASE_03"

# Step 3: Pull data from Oracle to Pandas DataFrames
base1 = read_oracle_data(query1, **config)
print('TBL_PL_BASE_01 Done')

base2 = read_oracle_data(query2, **config)
print('TBL_PL_BASE_02 Done')

base3 = read_oracle_data(query3, **config)
print('TBL_PL_BASE_03 Done')


/tmp/ipykernel_3219/133235839.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=conn)


TBL_PL_BASE_01 Done
TBL_PL_BASE_02 Done
TBL_PL_BASE_03 Done


In [13]:
base1['MSISDN'] = base1['MSISDN'].astype(int)
base2['MSISDN_1'] = base2['MSISDN_1'].astype(int)
base3['MSISDN_2'] = base3['MSISDN_2'].astype(int)

In [14]:
base = base1.merge(base2, 'left',left_on = 'MSISDN',right_on = 'MSISDN_1') \
                .merge(base3, how = 'left',left_on = 'MSISDN',right_on = 'MSISDN_2')

In [15]:
del base1; del base2; del base3

In [16]:
#Lowercase all column names
base.columns = base.columns.str.lower()
#One-hot encode categorical variables
base = pd.get_dummies(base, columns=['circle','region','rchg_chnl','segment_9box'])
# #Lowercase all column names
base.columns = base.columns.str.lower()

In [17]:
#Removing outliers
base = base[base["recharge_amount"] <= base["recharge_amount"].quantile(0.98)]
base = base[base["total_dstr"] <= base["total_dstr"].quantile(0.99)]

#Encoding categorical variables
label_le = preprocessing.LabelEncoder()
base['label'] = label_le.fit_transform(base['amount'])


#Feature engineering
column_diff_01 = ['dstr_change_01','mou_change_01','vol_change_01']
column_diff_12 = ['dstr_change_12','mou_change_12','vol_change_12']
column_0 = ['total_dstr','mo_mou','vol_mb']
column_1 = ['total_dstr_1','mo_mou_1','vol_mb_1']
column_2 = ['total_dstr_2','mo_mou_2','vol_mb_2']

for index,name in enumerate(column_diff_01):
    base[name] = (base[column_0[index]]-base[column_1[index]]).fillna(base[column_0[index]])

for index,name in enumerate(column_diff_12):
    base[name] = (base[column_1[index]]-base[column_2[index]]).fillna(base[column_1[index]].fillna(0))

In [18]:
#Rank in terms of pack rev to make distinct transaction
base['pack_rank'] = base.groupby(['msisdn'])['hit'].rank(method="first",ascending = False)
base=base.astype({"pack_rank": int})

In [19]:
#Fill null values with 0
for i in base.columns[base.isnull().any(axis=0)]:     
    base[i].fillna(0,inplace=True)

/tmp/ipykernel_3219/649223126.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  base[i].fillna(0,inplace=True)


In [20]:
#One hot encoding
srvcs = ['srvc1','srvc2','srvc3','srvc4','srvc5','srvc6','srvc7','srvc8']

for item in srvcs:
    base.loc[ (base[item]== 1) | (base[item]== 3) ,item] = 1
    base.loc[ (base[item]== 2) | (base[item]== 4) ,item] = 0


In [22]:
# Compress and save with gzip
with gzip.open('artifacts/pl_base.pkl.gz', 'wb', compresslevel=4) as f:
    pickle.dump(base, f, protocol=4)


In [5]:
# #loading scalar data
with gzip.open('artifacts/pl_base.pkl.gz', 'rb') as f:
    base=pickle.load(f)



----------------------------------------------

## Pack Taker

In [6]:
cols = ['msisdn', 'total_dstr', 'voicerev_total', 'datarev_total',
       'mixed_bundle_rev', 'voicerev_trig', 'datarev_trig', 'vol_mb', 'mo_mou',
       'recharge_cnt', 'recharge_amount', 'recharge_max', 'data_rg_days',
       'voice_rg_days', 'smartphone', 'subs_all_90d', 
       'days_since_last_rcrg', 'total_dstr_1', 'vol_mb_1',
       'mo_mou_1', 'recharge_cnt_1', 'voicerev_total_1', 'datarev_total_1',
       'mixed_bundle_rev_1', 'voicerev_trig_1', 'datarev_trig_1',
       'total_dstr_2', 'vol_mb_2', 'mo_mou_2', 'recharge_cnt_2',
       'voicerev_total_2', 'datarev_total_2', 'mixed_bundle_rev_2',
       'voicerev_trig_2', 'datarev_trig_2','recharge_max_1', 'recharge_max_2', 
       'dstr_change_01', 'mou_change_01', 'vol_change_01', 'dstr_change_12',
       'mou_change_12', 'vol_change_12','amount1','amount2','amount3','amount4','amount5','amount6','amount7','amount8',
       'srvc1', 'srvc2', 'srvc3', 'srvc4', 'srvc5', 'srvc6', 'srvc7', 'srvc8','circle_chittagong','circle_dhaka',
       'circle_khulna','circle_mymensingh','circle_rajshahi','circle_sylhet','circle_unknown','rchg_chnl_01. freq_retail_only',
       'rchg_chnl_02. freq_retail_dominant','rchg_chnl_03. freq_mixed','rchg_chnl_04. freq_digital_dominant','rchg_chnl_05. freq_digital_only',
       'rchg_chnl_unknown', 'segment_9box_data engaged','segment_9box_data only','segment_9box_data optimizer','segment_9box_low engaged','segment_9box_mature user',
       'segment_9box_premium','segment_9box_silent base','segment_9box_ultra premium','segment_9box_voice only']



features = ['total_dstr', 'voicerev_total', 'datarev_total',
       'mixed_bundle_rev', 'voicerev_trig', 'datarev_trig', 'vol_mb', 'mo_mou',
       'recharge_cnt', 'recharge_amount', 'recharge_max', 'data_rg_days',
       'voice_rg_days','days_since_last_rcrg','total_dstr_1', 'vol_mb_1',       
       'mo_mou_1', 'recharge_cnt_1', 'voicerev_total_1', 'datarev_total_1',
       'mixed_bundle_rev_1', 'voicerev_trig_1', 'datarev_trig_1',
       'total_dstr_2', 'vol_mb_2', 'mo_mou_2', 'recharge_cnt_2',
       'voicerev_total_2', 'datarev_total_2', 'mixed_bundle_rev_2',
       'voicerev_trig_2', 'datarev_trig_2','recharge_max_1', 'recharge_max_2',
       'dstr_change_01', 'mou_change_01', 'vol_change_01', 'dstr_change_12',
       'mou_change_12', 'vol_change_12','amount1','amount2','amount3','amount4','amount5','amount6','amount7','amount8',
        'srvc1', 'srvc2', 'srvc3', 'srvc4', 'srvc5', 'srvc6', 'srvc7', 'srvc8','circle_chittagong','circle_dhaka',
       'circle_khulna','circle_mymensingh','circle_rajshahi','circle_sylhet','circle_unknown','rchg_chnl_01. freq_retail_only',
       'rchg_chnl_02. freq_retail_dominant','rchg_chnl_03. freq_mixed','rchg_chnl_04. freq_digital_dominant','rchg_chnl_05. freq_digital_only',
       'rchg_chnl_unknown','segment_9box_data engaged','segment_9box_data only','segment_9box_data optimizer','segment_9box_low engaged','segment_9box_mature user',
       'segment_9box_premium','segment_9box_silent base','segment_9box_ultra premium','segment_9box_voice only']
target = ['label']

### Model Train

In [7]:
#One hot encoding
base['label_one_hot'] = list(to_categorical(base['label'])) 

/tmp/ipykernel_81722/1952599377.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  base['label_one_hot'] = list(to_categorical(base['label']))


In [8]:
base_sample = base[base.pack_rank==1].sample(frac = 0.7, random_state = 111)

In [9]:
#Test train split
train_ind, test_ind = train_test_split(base_sample.index, test_size = 0.3, random_state = 123)
train_x = base_sample.loc[train_ind,features].astype(np.float32)
train_y = base_sample.loc[train_ind,'label_one_hot']

test_x = base_sample.loc[test_ind,features].astype(np.float32)
test_y = base_sample.loc[test_ind,'label_one_hot']

In [10]:
#Feature 
sc = StandardScaler().fit(train_x)
train_x_scaled = sc.transform(train_x)
test_x_scaled = sc.transform(test_x)

In [ ]:
# Enable XLA optimization
tf.config.optimizer.set_jit(True)

#Check for GPU
gpus = tf.config.list_physical_devices('GPU')
device = 'GPU:0'

print(f"Using device: {device}")

# Define the model
def build_model():
    nodes = 2048
    model = Sequential([
        Dense(nodes, activation='relu', input_shape=(len(features),)),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(51, activation='softmax')
    ])
    return model

# Build model inside device context
with tf.device(device):
    model = build_model()

# Model Summary
model.summary()

# Convert training data to tensors
train_x_tensor = tf.convert_to_tensor(train_x_scaled, dtype=tf.float32)
train_y_tensor = tf.convert_to_tensor(np.array(train_y.values.tolist()), dtype=tf.float16)

# Hyperparameters and Training inside device context
with tf.device(device):
    learn_rate = 0.0001
    optim = tf.keras.optimizers.Adam(learning_rate=learn_rate)
    
    model.compile(
        loss='categorical_crossentropy',
        optimizer=optim,
        metrics=['accuracy']
    )
    
    file = 'artifacts/pl_reco_taker.h5'
    checkpointer = ModelCheckpoint(
        filepath=file,
        verbose=1,
        save_best_only=True
    )
    
    model.fit(
        train_x_tensor, 
        train_y_tensor, 
        epochs=20, 
        batch_size=512, 
        verbose=1, 
        validation_split=0.3, 
        callbacks=[checkpointer]
    )

Using device: GPU:0


/opt/conda/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1762787066.931971   81722 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43500 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:38:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 2048)           │       161,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2048)           │     4,196,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2048)           │     4,196,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 51)             │       104,499 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,658,995 (33.03 MB)

...


Epoch 1: val_loss improved from None to 1.46218, saving model to artifacts/pl_reco_taker.h5


9223/9223 ━━━━━━━━━━━━━━━━━━━━ 129s 13ms/step - accuracy: 0.5685 - loss: 1.5489 - val_accuracy: 0.5916 - val_loss: 1.4622
Epoch 2/20
9220/9223 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5893 - loss: 1.4761
Epoch 2: val_loss improved from 1.46218 to 1.43192, saving model to artifacts/pl_reco_taker.h5


9223/9223 ━━━━━━━━━━━━━━━━━━━━ 106s 11ms/step - accuracy: 0.5917 - loss: 1.4697 - val_accuracy: 0.6014 - val_loss: 1.4319
Epoch 3/20
9217/9223 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5975 - loss: 1.4503
Epoch 3: val_loss improved from 1.43192 to 1.41772, saving model to artifacts/pl_reco_taker.h5


9223/9223 ━━━━━━━━━━━━━━━━━━━━ 106s 11ms/step - accuracy: 0.5989 - loss: 1.4465 - val_accuracy: 0.6062 - val_loss: 1.4177
Epoch 4/20
9217/9223 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6025 - loss: 1.4339
Epoch 4: val_loss improved from 1.41772 to 1.41163, saving model to artifacts/pl_reco_taker.h5


9223/9223 ━━━━━━━━━━━━━━━━━━━━ 105s 11ms/step - accuracy: 0.6030 - loss: 1.4330 - val_accuracy: 0.6081 - val_loss: 1.4116
Epoch 5/20
9222/9223 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6053 - loss: 1.4241

In [48]:
#del train_x_tensor; del train_y_tensor;
# del model;

In [5]:
#Saving base file
# with open('artifacts/pl_scaler_taker.pkl','wb') as f:
#     pickle.dump(sc, f, protocol=4)

In [30]:
del train_x_tensor; del train_y_tensor;

In [31]:
test_x = base.loc[test_ind,cols]
#Ranking by amount
test_x['pack_rank'] = (test_x.groupby('msisdn')['total_dstr'].rank(method="first",ascending = False))
test_x = test_x.astype({"pack_rank": int})
#Selecting msisdn with top rev
base_infer = test_x[test_x.pack_rank==1][cols]
base_infer_scaled = sc.transform(base_infer[features])
# class_names = label_le.classes_
class_names = base.amount.unique()
class_names.sort()
predictions = model.predict(tf.convert_to_tensor(base_infer_scaled, dtype=tf.float32))
print('Model Scoring')
for index,name in enumerate(class_names):
    print('Predicting for:', index)
    base_infer[name] = predictions[:,index]

2025-11-10 11:23:48.678591: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-10 11:23:48.678910: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-10 11:23:50.614130: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_32', 12 bytes spill stores, 12 bytes spill loads

2025-11-10 11:23:50.774740: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : R

2025-11-10 11:23:51.410358: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_32', 116 bytes spill stores, 116 bytes spill loads

2025-11-10 11:23:51.490674: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_32', 232 bytes spill stores, 236 bytes spill loads

2025-11-10 11:23:51.809404: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_32', 332 bytes spill stores, 332 bytes spill loads

2025-11-10 11:23:52.481654: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_32', 10212 bytes spill stores, 10236 bytes spill loads



90333/90341 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

2025-11-10 11:30:10.067513: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-10 11:30:10.067941: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-10 11:30:11.895183: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_32', 12 bytes spill stores, 12 bytes spill loads

2025-11-10 11:30:12.125029: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : R

90341/90341 ━━━━━━━━━━━━━━━━━━━━ 387s 4ms/step
Model Scoring
Predicting for: 0
Predicting for: 1
Predicting for: 2
Predicting for: 3
Predicting for: 4
Predicting for: 5
Predicting for: 6
Predicting for: 7
Predicting for: 8
Predicting for: 9
Predicting for: 10
Predicting for: 11
Predicting for: 12
Predicting for: 13
Predicting for: 14
Predicting for: 15
Predicting for: 16
Predicting for: 17
Predicting for: 18
Predicting for: 19
Predicting for: 20
Predicting for: 21
Predicting for: 22
Predicting for: 23
Predicting for: 24
Predicting for: 25
Predicting for: 26
Predicting for: 27
Predicting for: 28
Predicting for: 29
Predicting for: 30
Predicting for: 31
Predicting for: 32
Predicting for: 33
Predicting for: 34
Predicting for: 35
Predicting for: 36
Predicting for: 37
Predicting for: 38
Predicting for: 39
Predicting for: 40
Predicting for: 41
Predicting for: 42
Predicting for: 43
Predicting for: 44
Predicting for: 45
Predicting for: 46
Predicting for: 47
Predicting for: 48
Predicting for: 49

In [32]:
# packs = label_le.classes_
packs = base.amount.unique()
packs.sort()
mpb_taker = base_infer.melt(id_vars = 'msisdn', value_vars = packs, var_name = 'pack', value_name = 'prob')
#Ranking by prob
mpb_taker['rank'] = (mpb_taker.groupby('msisdn')['prob'].rank(method="first",ascending = False))
mpb_taker = mpb_taker.astype({"rank": int})
#Selecting top 8 packs
top_eight_pred = mpb_taker[mpb_taker['rank'] <= 10]
top_pred = top_eight_pred.pivot(index='msisdn', columns='rank', values='pack')

In [55]:
# len(top_eight_pred.pack.unique())

### Test Data Precision

In [33]:
target_base = base.loc[test_ind,['msisdn','amount']]
target_base = base[base.msisdn.isin(target_base.msisdn.unique())][['msisdn','amount']]
pack_taker = pd.merge(top_eight_pred ,target_base, how='inner', left_on=['msisdn','pack'], right_on = ['msisdn','amount'])

In [6]:
# base_count = len(target_base.msisdn.unique())
# taker_count = len(pack_taker.msisdn.unique())

In [35]:
print('Precision: ',(taker_count/base_count)*100)

Precision:  96.58183873526373


## Non Taker Model

In [7]:
cols = ['msisdn', 'total_dstr', 'voicerev_total', 'datarev_total',
       'mixed_bundle_rev', 'voicerev_trig', 'datarev_trig', 'vol_mb', 'mo_mou',
       'recharge_cnt', 'recharge_amount', 'recharge_max', 'data_rg_days',
       'voice_rg_days', 'smartphone', 'subs_all_90d', 
       'days_since_last_rcrg', 'total_dstr_1', 'vol_mb_1',
       'mo_mou_1', 'recharge_cnt_1', 'voicerev_total_1', 'datarev_total_1',
       'mixed_bundle_rev_1', 'voicerev_trig_1', 'datarev_trig_1',
       'total_dstr_2', 'vol_mb_2', 'mo_mou_2', 'recharge_cnt_2',
       'voicerev_total_2', 'datarev_total_2', 'mixed_bundle_rev_2',
       'voicerev_trig_2', 'datarev_trig_2','recharge_max_1', 'recharge_max_2', 'label', 
       'dstr_change_01', 'mou_change_01', 'vol_change_01', 'dstr_change_12',
       'mou_change_12', 'vol_change_12','circle_chittagong','circle_dhaka',
       'circle_khulna','circle_mymensingh','circle_rajshahi','circle_sylhet','circle_unknown','rchg_chnl_01. freq_retail_only',
       'rchg_chnl_02. freq_retail_dominant','rchg_chnl_03. freq_mixed','rchg_chnl_04. freq_digital_dominant','rchg_chnl_05. freq_digital_only',
       'rchg_chnl_unknown', 'segment_9box_data engaged','segment_9box_data only','segment_9box_data optimizer','segment_9box_low engaged','segment_9box_mature user',
       'segment_9box_premium','segment_9box_silent base','segment_9box_ultra premium','segment_9box_voice only']


features = ['total_dstr', 'voicerev_total', 'datarev_total',
       'mixed_bundle_rev', 'voicerev_trig', 'datarev_trig', 'vol_mb', 'mo_mou',
       'recharge_cnt', 'recharge_amount', 'recharge_max', 'data_rg_days',
       'voice_rg_days','days_since_last_rcrg','total_dstr_1', 'vol_mb_1',       
       'mo_mou_1', 'recharge_cnt_1', 'voicerev_total_1', 'datarev_total_1',
       'mixed_bundle_rev_1', 'voicerev_trig_1', 'datarev_trig_1',
       'total_dstr_2', 'vol_mb_2', 'mo_mou_2', 'recharge_cnt_2',
       'voicerev_total_2', 'datarev_total_2', 'mixed_bundle_rev_2',
       'voicerev_trig_2', 'datarev_trig_2','recharge_max_1', 'recharge_max_2',
       'dstr_change_01', 'mou_change_01', 'vol_change_01', 'dstr_change_12',
       'mou_change_12', 'vol_change_12','circle_chittagong','circle_dhaka',
       'circle_khulna','circle_mymensingh','circle_rajshahi','circle_sylhet','circle_unknown','rchg_chnl_01. freq_retail_only',
       'rchg_chnl_02. freq_retail_dominant','rchg_chnl_03. freq_mixed','rchg_chnl_04. freq_digital_dominant','rchg_chnl_05. freq_digital_only',
       'rchg_chnl_unknown']
target = ['label']

In [8]:
# #One hot encoding
base['label_one_hot'] = list(to_categorical(base['label'])) 

In [9]:
base_sample = base.sample(frac = 1, random_state = 111)

In [10]:
# #Test train split
train_ind, test_ind = train_test_split(base_sample[base_sample.pack_rank==1].index, test_size = 0.2, random_state = 123)
train_x = base_sample.loc[train_ind,features]
train_y = base_sample.loc[train_ind,'label_one_hot']

test_x = base_sample.loc[test_ind,features]
test_y = base_sample.loc[test_ind,'label_one_hot']

In [11]:
# #Feature Scaling
sc = StandardScaler().fit(train_x)
train_x_scaled = sc.transform(train_x)
test_x_scaled = sc.transform(test_x)

In [13]:
# Enable XLA optimization
# tf.config.optimizer.set_jit(True)
# tf.config.set_visible_devices([], 'GPU')

# Check for GPU
# gpus = tf.config.list_physical_devices('GPU')
device =  '/GPU:0'
print(f"Using device: {device}")

# Define the model
def build_model():
    nodes = 2048
    model = Sequential([
        Dense(nodes, activation='relu', input_shape=(train_x_scaled.shape[1],)),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(51, activation='softmax')
    ])
    return model

# Build model inside device context
with tf.device(device):
    model_non_taker = build_model()

# Model Summary
model_non_taker.summary()

# Convert training data to tensors
train_x_tensor = tf.convert_to_tensor(train_x_scaled, dtype=tf.float32)
train_y_tensor = tf.convert_to_tensor(np.array(train_y.values.tolist()), dtype=tf.float16)

# Hyperparameters and Training inside device context
with tf.device(device):
    learn_rate = 0.0002
    optim = tf.keras.optimizers.Adam(learning_rate=learn_rate)
    
    model_non_taker.compile(
        loss='categorical_crossentropy',
        optimizer=optim,
        metrics=['accuracy']
    )
    
    file = 'artifacts/pl_reco_non_taker.h5'
    checkpointer = ModelCheckpoint(
        filepath=file,
        verbose=1,
        save_best_only=True
    )
    
    model_non_taker.fit(
        train_x_tensor, 
        train_y_tensor, 
        epochs=20, 
        batch_size=1024, 
        verbose=1, 
        validation_split=0.3, 
        callbacks=[checkpointer]
    )

Using device: /CPU:0
Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
dense (Dense)                (None, 2048)              110592    
_________________________________________________________________
dropout (Dropout)            (None, 2048)              0         
_________________________________________________________________
dense_1 (Dense)              (None, 2048)              4196352   
_________________________________________________________________
dropout_1 (Dropout)          (None, 2048)              0         
_________________________________________________________________
dense_2 (Dense)              (None, 2048)              4196352   
_________________________________________________________________
dropout_2 (Dropout)          (None, 2048)              0         
_________________________________________________________________
dense_3 (Dense)              (None,

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



7360/7360 [==============================] - 979s 133ms/step - loss: 1.9800 - accuracy: 0.4653 - val_loss: 1.9472 - val_accuracy: 0.4741

Epoch 00006: val_loss improved from 1.95240 to 1.94718, saving model to pl_reco_non_taker.hdf5
Epoch 7/20
7360/7360 [==============================] - 951s 129ms/step - loss: 1.9738 - accuracy: 0.4671 - val_loss: 1.9433 - val_accuracy: 0.4757

Epoch 00007: val_loss improved from 1.94718 to 1.94327, saving model to pl_reco_non_taker.hdf5
Epoch 8/20
7360/7360 [==============================] - 983s 134ms/step - loss: 1.9689 - accuracy: 0.4685 - val_loss: 1.9413 - val_accuracy: 0.4766

Epoch 00008: val_loss improved from 1.94327 to 1.94134, saving model to pl_reco_non_taker.hdf5
Epoch 9/20
7360/7360 [==============================] - 964s 131ms/step - loss: 1.9644 - accuracy: 0.4698 - val_loss: 1.9378 - val_accuracy: 0.4774

Epoch 00009: val_loss improved from 1.94134 to 1.93783, saving model to pl_reco_non_taker.hdf5
Epoch 10/20
7360/7360 [============

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



7360/7360 [==============================] - 965s 131ms/step - loss: 1.9489 - accuracy: 0.4740 - val_loss: 1.9277 - val_accuracy: 0.4812

Epoch 00014: val_loss improved from 1.92984 to 1.92769, saving model to pl_reco_non_taker.hdf5
Epoch 15/20
7273/7360 [============================>.] - ETA: 10s - loss: 1.9464 - accuracy: 0.4745

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



5925/7360 [=======================>......] - ETA: 2:50 - loss: 1.9443 - accuracy: 0.4750

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



7360/7360 [==============================] - 954s 130ms/step - loss: 1.9444 - accuracy: 0.4751 - val_loss: 1.9289 - val_accuracy: 0.4812

Epoch 00016: val_loss did not improve from 1.92769
Epoch 17/20
7360/7360 [==============================] - 952s 129ms/step - loss: 1.9421 - accuracy: 0.4756 - val_loss: 1.9271 - val_accuracy: 0.4818

Epoch 00017: val_loss improved from 1.92769 to 1.92714, saving model to pl_reco_non_taker.hdf5
Epoch 18/20
7360/7360 [==============================] - 952s 129ms/step - loss: 1.9401 - accuracy: 0.4762 - val_loss: 1.9262 - val_accuracy: 0.4819

Epoch 00018: val_loss improved from 1.92714 to 1.92624, saving model to pl_reco_non_taker.hdf5
Epoch 19/20
7360/7360 [==============================] - 946s 128ms/step - loss: 1.9382 - accuracy: 0.4766 - val_loss: 1.9255 - val_accuracy: 0.4824

Epoch 00019: val_loss improved from 1.92624 to 1.92553, saving model to pl_reco_non_taker.hdf5
Epoch 20/20
7360/7360 [==============================] - 942s 128ms/step - l

In [85]:
#del train_x_tensor; del train_y_tensor; del model_non_taker;

In [14]:
test_x = base.loc[test_ind,cols]
#Ranking by amount
test_x['pack_rank'] = (test_x.groupby('msisdn')['total_dstr'].rank(method="first",ascending = False))
test_x = test_x.astype({"pack_rank": int})
#Selecting msisdn with top rev
base_infer = test_x[test_x.pack_rank==1][cols]
# base_infer_scaled = base_infer[features]
base_infer_scaled = sc.transform(base_infer[features])
# class_names = label_le.classes_
class_names = base.amount.unique()
class_names.sort()
predictions = model_non_taker.predict(base_infer_scaled)
print('Model Scoring')
for index,name in enumerate(class_names):
    print('Predicting for:', index)
    base_infer[name] = predictions[:,index]

Model Scoring
Predicting for: 0
Predicting for: 1
Predicting for: 2
Predicting for: 3
Predicting for: 4
Predicting for: 5
Predicting for: 6
Predicting for: 7
Predicting for: 8
Predicting for: 9
Predicting for: 10
Predicting for: 11
Predicting for: 12
Predicting for: 13
Predicting for: 14
Predicting for: 15
Predicting for: 16
Predicting for: 17
Predicting for: 18
Predicting for: 19
Predicting for: 20
Predicting for: 21
Predicting for: 22
Predicting for: 23
Predicting for: 24
Predicting for: 25
Predicting for: 26
Predicting for: 27
Predicting for: 28
Predicting for: 29
Predicting for: 30
Predicting for: 31
Predicting for: 32
Predicting for: 33
Predicting for: 34
Predicting for: 35
Predicting for: 36
Predicting for: 37
Predicting for: 38
Predicting for: 39
Predicting for: 40
Predicting for: 41
Predicting for: 42
Predicting for: 43
Predicting for: 44
Predicting for: 45
Predicting for: 46
Predicting for: 47
Predicting for: 48
Predicting for: 49
Predicting for: 50
Predicting for: 51
Predicti

In [15]:
# packs = label_le.classes_
packs = base.amount.unique()
packs.sort()
mpb_taker = base_infer.melt(id_vars = 'msisdn', value_vars = packs, var_name = 'pack', value_name = 'prob')
#Ranking by prob
mpb_taker['rank'] = (mpb_taker.groupby('msisdn')['prob'].rank(method="first",ascending = False))
mpb_taker = mpb_taker.astype({"rank": int})
#Selecting top 8 packs
top_eight_pred = mpb_taker[mpb_taker['rank'] <= 10]
top_pred = top_eight_pred.pivot(index='msisdn', columns='rank', values='pack')

In [16]:
target_base = base.loc[test_ind,['msisdn','amount']]
target_base = base[base.msisdn.isin(target_base.msisdn.unique())][['msisdn','amount']]
# pack_taker = pd.merge(top_eight_pred[top_eight_pred['prob']>0.02]  ,target_base, how='inner', left_on=['msisdn','pack'], right_on = ['msisdn','amount'])
pack_taker = pd.merge(top_eight_pred  ,target_base, how='inner', left_on=['msisdn','pack'], right_on = ['msisdn','amount'])

In [17]:
base_count = len(target_base.msisdn.unique())
taker_count = len(pack_taker.msisdn.unique())

In [18]:
print('Precision: ',(taker_count/base_count)*100)

Precision:  91.55499046597102


In [19]:
# #saving scalar data
with open('artifacts/pl_test_scalar.pkl','wb') as f:
    pickle.dump(sc, f)


## Base Generation

### Pack Taker

In [ ]:
# Step 2: Execute the SQL queries to retrieve data
query1 = "SELECT * FROM TBL_PL_BASE_INFER_01"
query2 = "SELECT * FROM TBL_PL_BASE_INFER_02"
query3 = "SELECT * FROM TBL_PL_BASE_INFER_03"

# Step 3: Pull data from Oracle to Pandas DataFrames
base1 = read_oracle_data(query1, **config)
print('TBL_PL_BASE_01 Done')

base2 = read_oracle_data(query2, **config)
print('TBL_PL_BASE_02 Done')

base3 = read_oracle_data(query3, **config)
print('TBL_PL_BASE_03 Done')


/tmp/ipykernel_199498/133235839.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, con=conn)


TBL_PL_BASE_01 Done
TBL_PL_BASE_02 Done


In [10]:
base = base1.merge(base2, 'left',left_on = 'MSISDN',right_on = 'MSISDN_1') \
                .merge(base3, how = 'left',left_on = 'MSISDN',right_on = 'MSISDN_2')

In [11]:
del base1; del base2; del base3

In [12]:
#Lowercase all column names
base.columns = base.columns.str.lower()
#One-hot encode categorical variables
base = pd.get_dummies(base, columns=['circle','region','rchg_chnl','segment_9box'])
# #Lowercase all column names
base.columns = base.columns.str.lower()

In [13]:
#Lowercase all column names
base.columns = base.columns.str.lower()
#Removing outliers
# base = base[base["hit"] < base["hit"].quantile(0.97)]
base = base[base["recharge_amount"] < base["recharge_amount"].quantile(0.98)]
base = base[base["total_dstr"] < base["total_dstr"].quantile(0.99)]

#Feature engineering
column_diff_01 = ['dstr_change_01','mou_change_01','vol_change_01']
column_diff_12 = ['dstr_change_12','mou_change_12','vol_change_12']
column_0 = ['total_dstr','mo_mou','vol_mb']
column_1 = ['total_dstr_1','mo_mou_1','vol_mb_1']
column_2 = ['total_dstr_2','mo_mou_2','vol_mb_2']

for index,name in enumerate(column_diff_01):
    base[name] = (base[column_0[index]]-base[column_1[index]]).fillna(base[column_0[index]])

for index,name in enumerate(column_diff_12):
    base[name] = (base[column_1[index]]-base[column_2[index]]).fillna(base[column_1[index]].fillna(0))

In [ ]:
#Fill null values with 0
for i in base.columns[base.isnull().any(axis=0)]:     
    base[i].fillna(0,inplace=True)

/tmp/ipykernel_199498/649223126.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  base[i].fillna(0,inplace=True)
/tmp/ipykernel_199498/649223126.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  base[i].fillna(0,inplace=True)


In [15]:
#One hot encoding
srvcs = ['srvc1','srvc2','srvc3','srvc4','srvc5','srvc6','srvc7','srvc8']

for item in srvcs:
    base.loc[ (base[item]== 1) | (base[item]== 3) ,item] = 1
    base.loc[ (base[item]== 2) | (base[item]== 4) ,item] = 0

In [ ]:
#Saving base file
# with open('artifacts/pl_base_infer.pkl','wb') as f:
#     pickle.dump(base, f, protocol=4)

# Compress and save with gzip
with gzip.open('artifacts/pl_base_infer.gz', 'wb', compresslevel=4) as f:
    pickle.dump(base, f, protocol=4)

In [6]:
#Loading base file
with gzip.open('artifacts/pl_base_infer.gz','rb') as f:
    base=pickle.load(f)
    


### Taker Prediction

In [20]:
cols = ['msisdn', 'total_dstr', 'voicerev_total', 'datarev_total',
       'mixed_bundle_rev', 'voicerev_trig', 'datarev_trig', 'vol_mb', 'mo_mou',
       'recharge_cnt', 'recharge_amount', 'recharge_max', 'data_rg_days',
       'voice_rg_days', 'smartphone', 'subs_all_90d', 
       'days_since_last_rcrg', 'total_dstr_1', 'vol_mb_1',
       'mo_mou_1', 'recharge_cnt_1', 'voicerev_total_1', 'datarev_total_1',
       'mixed_bundle_rev_1', 'voicerev_trig_1', 'datarev_trig_1',
       'total_dstr_2', 'vol_mb_2', 'mo_mou_2', 'recharge_cnt_2',
       'voicerev_total_2', 'datarev_total_2', 'mixed_bundle_rev_2',
       'voicerev_trig_2', 'datarev_trig_2','recharge_max_1', 'recharge_max_2', 
       'dstr_change_01', 'mou_change_01', 'vol_change_01', 'dstr_change_12',
       'mou_change_12', 'vol_change_12','amount1','amount2','amount3','amount4','amount5','amount6','amount7','amount8',
       'srvc1', 'srvc2', 'srvc3', 'srvc4', 'srvc5', 'srvc6', 'srvc7', 'srvc8','circle_chittagong','circle_dhaka',
       'circle_khulna','circle_mymensingh','circle_rajshahi','circle_sylhet','circle_unknown','rchg_chnl_01. freq_retail_only',
       'rchg_chnl_02. freq_retail_dominant','rchg_chnl_03. freq_mixed','rchg_chnl_04. freq_digital_dominant','rchg_chnl_05. freq_digital_only',
       'rchg_chnl_unknown', 'segment_9box_data engaged','segment_9box_data only','segment_9box_data optimizer','segment_9box_low engaged','segment_9box_mature user',
       'segment_9box_premium','segment_9box_silent base','segment_9box_ultra premium','segment_9box_voice only']



features = ['total_dstr', 'voicerev_total', 'datarev_total',
       'mixed_bundle_rev', 'voicerev_trig', 'datarev_trig', 'vol_mb', 'mo_mou',
       'recharge_cnt', 'recharge_amount', 'recharge_max', 'data_rg_days',
       'voice_rg_days','days_since_last_rcrg','total_dstr_1', 'vol_mb_1',       
       'mo_mou_1', 'recharge_cnt_1', 'voicerev_total_1', 'datarev_total_1',
       'mixed_bundle_rev_1', 'voicerev_trig_1', 'datarev_trig_1',
       'total_dstr_2', 'vol_mb_2', 'mo_mou_2', 'recharge_cnt_2',
       'voicerev_total_2', 'datarev_total_2', 'mixed_bundle_rev_2',
       'voicerev_trig_2', 'datarev_trig_2','recharge_max_1', 'recharge_max_2',
       'dstr_change_01', 'mou_change_01', 'vol_change_01', 'dstr_change_12',
       'mou_change_12', 'vol_change_12','amount1','amount2','amount3','amount4','amount5','amount6','amount7','amount8',
        'srvc1', 'srvc2', 'srvc3', 'srvc4', 'srvc5', 'srvc6', 'srvc7', 'srvc8','circle_chittagong','circle_dhaka',
       'circle_khulna','circle_mymensingh','circle_rajshahi','circle_sylhet','circle_unknown','rchg_chnl_01. freq_retail_only',
       'rchg_chnl_02. freq_retail_dominant','rchg_chnl_03. freq_mixed','rchg_chnl_04. freq_digital_dominant','rchg_chnl_05. freq_digital_only',
       'rchg_chnl_unknown','segment_9box_data engaged','segment_9box_data only','segment_9box_data optimizer','segment_9box_low engaged','segment_9box_mature user',
       'segment_9box_premium','segment_9box_silent base','segment_9box_ultra premium','segment_9box_voice only']
target = ['label']

In [21]:
#Loading base file
with open('artifacts/pl_scaler_taker.pkl','rb') as f:
    sc=pickle.load(f)

In [ ]:
# Disable XLA optimization
tf.config.optimizer.set_jit(False)
# tf.config.set_visible_devices([], 'GPU')
# Check for GPU
# gpus = tf.config.list_physical_devices('GPU')
device =  '/GPU:0'
# print(f"Using device: {device}")

# Define the model
def build_model():
    nodes = 2048
    model = Sequential([
        Dense(nodes, activation='relu', input_shape=(len(features),)),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(51, activation='softmax')
    ])
    return model

# Build model inside device context
with tf.device(device):
    model = build_model()


#Loading taker model
model.load_weights('artifacts/pl_reco_taker.h5')

I0000 00:00:1763195285.669578  199498 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43500 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:38:00.0, compute capability: 8.9
/opt/conda/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
# #Scaling features
# base_infer = base[base['pack_flag'] == 'TAKER'][cols]
# base_infer_scaled = sc.transform(base_infer.loc[:,features])


# #Generating predictions
# start_pos = 0
# batch_size = 5000000
# max_size = len(base_infer.msisdn)
# predictions =np.empty((0,51), int)
# print('Prediction generation')
# while start_pos < max_size:
#     print('Generating from',start_pos,'to',start_pos+batch_size)
#     predictions = np.vstack([predictions,model.predict(base_infer_scaled[start_pos:start_pos+batch_size])])
#     start_pos += batch_size

# class_names=[29,	39,	49,	98,	99,	118,	119,	148,	158,	179,	197,	198,	208,	217,	
#             219,	228,	242,	249,	258,	279,	308,	309,	319,	419,	499,	509,	510,	
#             519,	598,	599,	639,	698,	699,	729,	798,	818,	898,	988,	989,	997,	999,
# 	        1099,	1148,	1198,	1199,	1493,	1497,	1498,	1649,	2149,	2997]
# print('Model Scoring')
# for index,name in enumerate(class_names):
#     print('Predicting for:', index)
#     base_infer[name] = predictions[:,index]

In [ ]:
base.pack_flag.unique()

array(['NON_TAKER'], dtype=object)

In [21]:
#Saving base file
# Compress and save with gzip
with gzip.open('artifacts/pl_base_infer_pred.gz', 'wb', compresslevel=4) as f:
    pickle.dump(base_infer, f, protocol=4)

In [10]:
# #loading base file
with gzip.open('artifacts/pl_base_infer_pred.gz', 'rb') as f:
    base_infer = pickle.load(f)

### Non Pack Taker


In [9]:
cols = ['msisdn', 'total_dstr', 'voicerev_total', 'datarev_total',
       'mixed_bundle_rev', 'voicerev_trig', 'datarev_trig', 'vol_mb', 'mo_mou',
       'recharge_cnt', 'recharge_amount', 'recharge_max', 'data_rg_days',
       'voice_rg_days', 'smartphone', 'subs_all_90d', 
       'days_since_last_rcrg', 'total_dstr_1', 'vol_mb_1',
       'mo_mou_1', 'recharge_cnt_1', 'voicerev_total_1', 'datarev_total_1',
       'mixed_bundle_rev_1', 'voicerev_trig_1', 'datarev_trig_1',
       'total_dstr_2', 'vol_mb_2', 'mo_mou_2', 'recharge_cnt_2',
       'voicerev_total_2', 'datarev_total_2', 'mixed_bundle_rev_2',
       'voicerev_trig_2', 'datarev_trig_2','recharge_max_1', 'recharge_max_2', 
       'dstr_change_01', 'mou_change_01', 'vol_change_01', 'dstr_change_12',
       'mou_change_12', 'vol_change_12','circle_chittagong','circle_dhaka',
       'circle_khulna','circle_mymensingh','circle_rajshahi','circle_sylhet','circle_unknown','rchg_chnl_01. freq_retail_only',
       'rchg_chnl_02. freq_retail_dominant','rchg_chnl_03. freq_mixed','rchg_chnl_04. freq_digital_dominant','rchg_chnl_05. freq_digital_only',
       'rchg_chnl_unknown', 'segment_9box_data engaged','segment_9box_data only','segment_9box_data optimizer','segment_9box_low engaged','segment_9box_mature user',
       'segment_9box_premium','segment_9box_silent base','segment_9box_ultra premium','segment_9box_voice only']


features = ['total_dstr', 'voicerev_total', 'datarev_total',
       'mixed_bundle_rev', 'voicerev_trig', 'datarev_trig', 'vol_mb', 'mo_mou',
       'recharge_cnt', 'recharge_amount', 'recharge_max', 'data_rg_days',
       'voice_rg_days','days_since_last_rcrg','total_dstr_1', 'vol_mb_1',       
       'mo_mou_1', 'recharge_cnt_1', 'voicerev_total_1', 'datarev_total_1',
       'mixed_bundle_rev_1', 'voicerev_trig_1', 'datarev_trig_1',
       'total_dstr_2', 'vol_mb_2', 'mo_mou_2', 'recharge_cnt_2',
       'voicerev_total_2', 'datarev_total_2', 'mixed_bundle_rev_2',
       'voicerev_trig_2', 'datarev_trig_2','recharge_max_1', 'recharge_max_2',
       'dstr_change_01', 'mou_change_01', 'vol_change_01', 'dstr_change_12',
       'mou_change_12', 'vol_change_12','circle_chittagong','circle_dhaka',
       'circle_khulna','circle_mymensingh','circle_rajshahi','circle_sylhet','circle_unknown','rchg_chnl_01. freq_retail_only',
       'rchg_chnl_02. freq_retail_dominant','rchg_chnl_03. freq_mixed','rchg_chnl_04. freq_digital_dominant','rchg_chnl_05. freq_digital_only',
       'rchg_chnl_unknown']

In [10]:
# Disable XLA optimization
tf.config.optimizer.set_jit(False)
# tf.config.set_visible_devices([], 'GPU')

# # Check for GPU
# # gpus = tf.config.list_physical_devices('GPU')
device =  '/GPU:0'
# print(f"Using device: {device}")

# Define the model
def build_model():
    nodes = 2048
    model = Sequential([
        Dense(nodes, activation='relu', input_shape=(len(features),)),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(nodes, activation='relu'),
        Dropout(0.3),
        Dense(51, activation='softmax')
    ])
    return model

# Build model inside device context
with tf.device(device):
    model_non_taker = build_model()


    
#Loading taker model
model_non_taker.load_weights('artifacts/pl_reco_non_taker.h5')

/opt/conda/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1763228676.121367  310305 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43500 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:38:00.0, compute capability: 8.9


In [11]:
# #saving scalar data
with open('artifacts/pl_test_scalar.pkl','rb') as f:
    sc = pickle.load(f)

In [ ]:
#Scaling features
base_non_infer = base[base['pack_flag'] == 'NON_TAKER'][cols]
base_non_infer_scaled = sc.transform(base_non_infer.loc[:,features])


#Generating predictions
start_pos = 0
batch_size = 20000000
max_size = len(base_non_infer.msisdn)
predictions =np.empty((0,51), int)
print('Prediction generation')
while start_pos < max_size:
    print('Generating from',start_pos,'to',start_pos+batch_size)
    predictions = np.vstack([predictions,model_non_taker.predict(base_non_infer_scaled[start_pos:start_pos+batch_size])])
    start_pos += batch_size

class_names=[29,	39,	49,	98,	99,	118,	119,	148,	158,	179,	197,	198,	208,	217,	
            219,	228,	242,	249,	258,	279,	308,	309,	319,	419,	499,	509,	510,	
            519,	598,	599,	639,	698,	699,	729,	798,	818,	898,	988,	989,	997,	999,
	        1099,	1148,	1198,	1199,	1493,	1497,	1498,	1649,	2149,	2997]
print('Model Scoring')
for index,name in enumerate(class_names):
    print('Predicting for:', index)
    base_non_infer[name] = predictions[:,index]

Prediction generation
Generating from 0 to 20000000


2025-11-15 17:46:36.670259: I external/local_xla/xla/service/service.cc:163] XLA service 0x7ef2dc004740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-15 17:46:36.670318: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA L40S, Compute Capability 8.9
2025-11-15 17:46:36.688530: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-15 17:46:36.767088: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91500
2025-11-15 17:46:36.873149: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-11-15 17:46:36.873579: I external/loca

In [35]:
with gzip.open('artifacts/pl_base_infer_non_pred.gz', 'wb', compresslevel=4) as f:
    pickle.dump(base_non_infer, f, protocol=4)

In [11]:
with gzip.open('artifacts/pl_base_infer_non_pred.gz', 'rb') as f:
    base_non_infer = pickle.load(f)

In [12]:
class_names=[29,	39,	49,	98,	99,	118,	119,	148,	158,	179,	197,	198,	208,	217,	
            219,	228,	242,	249,	258,	279,	308,	309,	319,	419,	499,	509,	510,	
            519,	598,	599,	639,	698,	699,	729,	798,	818,	898,	988,	989,	997,	999,
	        1099,	1148,	1198,	1199,	1493,	1497,	1498,	1649,	2149,	2997]
mpb_taker = base_infer.melt(id_vars = 'msisdn', value_vars = class_names, var_name = 'deno', value_name = 'prob')
mpb_non_taker = base_non_infer.melt(id_vars = 'msisdn', value_vars = class_names, var_name = 'deno', value_name = 'prob')

In [13]:
#Taker
mpb_taker['msisdn'] = mpb_taker['msisdn'].astype('category')
mpb_taker['prob'] = mpb_taker['prob'].astype('float16')
mpb_taker['rank'] = mpb_taker.groupby('msisdn', group_keys=False)['prob'] \
                             .rank(method='first', ascending=False).astype('int16')

#Non-taker
mpb_non_taker['msisdn'] = mpb_non_taker['msisdn'].astype('category')
mpb_non_taker['prob'] = mpb_non_taker['prob'].astype('float16')
mpb_non_taker['rank'] = mpb_non_taker.groupby('msisdn', group_keys=False)['prob'] \
                                     .rank(method='first', ascending=False).astype('int16')


/tmp/ipykernel_340684/1613213690.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mpb_taker['rank'] = mpb_taker.groupby('msisdn', group_keys=False)['prob'] \
/tmp/ipykernel_340684/1613213690.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mpb_non_taker['rank'] = mpb_non_taker.groupby('msisdn', group_keys=False)['prob'] \


In [14]:
base_taker_pred = mpb_taker[mpb_taker['rank'] <= 10]
base_nontaker_pred = mpb_non_taker[mpb_non_taker['rank'] <= 10]

In [15]:
# #saving taker/nontaker pred

with gzip.open('pl_base_taker_pred.gz', 'wb', compresslevel=4) as f:
# with open('pl_base_taker_pred.pkl','wb') as f:
    pickle.dump(base_taker_pred, f, protocol=4)
    
with gzip.open('pl_base_nontaker_pred.gz', 'wb', compresslevel=4) as f:  
# with open('pl_base_nontaker_pred.pkl','wb') as f:
    pickle.dump(base_nontaker_pred, f, protocol=4)

In [6]:
#loading base file
 
with gzip.open('pl_base_nontaker_pred.gz', 'rb') as f:
    base_nontaker_pred = pickle.load(f)

In [16]:
base_taker_pred['taker_flag'] = 1
base_nontaker_pred['taker_flag'] = 0

/tmp/ipykernel_340684/1222893820.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_taker_pred['taker_flag'] = 1
/tmp/ipykernel_340684/1222893820.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_nontaker_pred['taker_flag'] = 0


In [17]:
base_pred = pd.concat([base_taker_pred,base_nontaker_pred])

In [18]:
base_pred = base_pred.astype({"msisdn": int,"deno":int})

In [19]:
#Saving final base
# Compress and save with gzip
with gzip.open('artifacts/base_pred.gz', 'wb', compresslevel=4) as f:
    pickle.dump(base_pred, f, protocol=4)

#Loading final base
# with  gzip.open('artifacts/base_pred.gz','rb') as f:
#     base_pred = pickle.load(f)  

In [20]:
# Define the target table name
table_name = 'TBL_PL_PRED_202511'

# Map Pandas data types to Oracle SQL types
def get_oracle_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "NUMBER"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "NUMBER(1)"
    elif pd.api.types.is_string_dtype(dtype):
        return "VARCHAR2(255)"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "DATE"
    else:
        return "VARCHAR2(255)"  # Default to VARCHAR if unsure

# Construct the CREATE TABLE statement dynamically
create_table_sql = f"CREATE TABLE {table_name} ("
for column_name, dtype in base_pred.dtypes.items():
    oracle_type = get_oracle_type(dtype)
    create_table_sql += f"{column_name} {oracle_type}, "
create_table_sql = create_table_sql.rstrip(", ") + ")"  # Remove last comma and add closing parenthesis


In [21]:
# Establish the connection
connection = oracledb.connect(user="GPBI_MLOPS", password="Manag3r$Mlops", 
                                dsn="(DESCRIPTION=(ADDRESS=(PROTOCOL=TCP)(HOST=10.15.75.12)(PORT=1252))(CONNECT_DATA=(SERVER=dedicated)(SERVICE_NAME=EDWPROD)))", host="10.15.75.12",
                                port="1252",service_name="edw-100g-scan")


# Define the target table name
table_name = 'TBL_PL_PRED_202511'
batch_size = 1559752

# Insert DataFrame into Oracle Database using array binding for faster execution

try:
    # Open a cursor
    with connection.cursor() as cursor:
        try:
            cursor.execute(f"SELECT * FROM {table_name} WHERE 1=0")  # Quick check to see if table exists
        except oracledb.DatabaseError:
            # If the table does not exist, create it
            cursor.execute(create_table_sql)
            print(f"Table '{table_name}' created with structure based on DataFrame.")
            
        # Prepare for data insertion
        columns = ", ".join(base_pred.columns)
        placeholders = ", ".join([f":{i+1}" for i in range(len(base_pred.columns))])
        insert_sql = f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})"
        # Convert DataFrame to a list of tuples
        data_tuples = [tuple(row) for row in base_pred.itertuples(index=False, name=None)]
        count = 0
        for i in range(0, len(data_tuples), batch_size):
            # Insert in batches
            print(count)
            cursor.executemany(insert_sql, data_tuples[i:i + batch_size])
            connection.commit()  # Commit after each batch
            count += 1
        




    # Commit the transaction
    connection.commit()
    print(f"Data successfully exported to {table_name} in Oracle DB")
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    # Close the connection
    connection.close()

Table 'TBL_PL_PRED_202511' created with structure based on DataFrame.


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27